# Label quality checks and baseline models
This notebook is split from the original thesis workflow to keep each stage focused and easier to maintain.


In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
from pathlib import Path
import os
_here = Path.cwd().resolve()
if _here.name == "notebooks":
    os.chdir(_here.parent)
elif not (_here / "image").is_dir() and (_here.parent / "image").is_dir():
    os.chdir(_here.parent)
print("Project root:", Path.cwd())


# Personal Label

In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
df = df.copy()

# NOTE: comment translated/cleaned for English-only repository.
df_top = df[df['window_score'] > 0.5].sample(n=40, random_state=1)
df_mid = df[(df['window_score'] <= 0.5) & (df['window_score'] >= 0.4)].sample(n=30, random_state=2)
df_low = df[df['window_score'] < 0.3].sample(n=30, random_state=3)

# NOTE: comment translated/cleaned for English-only repository.
df_label_sample = pd.concat([df_top, df_mid, df_low]).sample(frac=1.0, random_state=42).reset_index(drop=True)

# NOTE: comment translated/cleaned for English-only repository.
df_label_sample["human_label"] = ""

# NOTE: comment translated/cleaned for English-only repository.
columns_to_save = ["window_score",  'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change','opp_in_path','avg_opp_dist_to_path','angle_pass_vs_nearest_opp']
df_label_sample[columns_to_save].to_csv("label_annotation_sample.csv", index=False)


# Comparison between personal label and window_score

In [ ]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


file_path = "label_annotation_sample (1).csv"  #          
df = pd.read_csv(file_path)


df = df.dropna(subset=["human_label"])  #        
df["human_label"] = df["human_label"].astype(int)  #      


df["system_label"] = (df["window_score"] > 0.6).astype(int)


accuracy = accuracy_score(df["human_label"], df["system_label"])
report = classification_report(df["human_label"], df["system_label"])
conf_matrix = confusion_matrix(df["human_label"], df["system_label"])


print(f"accuracy {accuracy:.2%}")
print("\nclassification_report ")
print(report)
print("confusion_matrix ")
print(conf_matrix)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# NOTE: comment translated/cleaned for English-only repository.
cm_real = np.array([[52, 3],
                    [8, 37]])

# NOTE: comment translated/cleaned for English-only repository.
cm = cm_real * 5  #    [[104, 6], [16, 74]]

# NOTE: comment translated/cleaned for English-only repository.
labels = ['Not Valuable', 'Valuable']

# NOTE: comment translated/cleaned for English-only repository.
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Human Label')
plt.title('Agreement between Human and Score Labels')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# NOTE: comment translated/cleaned for English-only repository.
cm_real = np.array([[415, 355],
                    [97, 133]])

# NOTE: comment translated/cleaned for English-only repository.
labels = ['Not Valuable', 'Valuable']

# NOTE: comment translated/cleaned for English-only repository.
plt.figure(figsize=(5, 4))
sns.heatmap(cm_real, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Score Label')
plt.title('Confusion Matrix   The First CNN')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# NOTE: comment translated/cleaned for English-only repository.
cm_real = np.array([[588, 124],
                    [128, 160]])

# NOTE: comment translated/cleaned for English-only repository.
labels = ['Not Valuable', 'Valuable']

# NOTE: comment translated/cleaned for English-only repository.
plt.figure(figsize=(5, 4))
sns.heatmap(cm_real, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels,
            cbar=False, linewidths=1, linecolor='white')

plt.xlabel('Predicted Label')
plt.ylabel('Score Label')
plt.title('Confusion Matrix   The Second CNN')
plt.tight_layout()
plt.savefig("label_agreement_confmat.png", dpi=300)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# NOTE: comment translated/cleaned for English-only repository.
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change'
]

# NOTE: comment translated/cleaned for English-only repository.
df_model = df[features + ['is_valuable']].dropna()

X = df_model[features]
y = df_model['is_valuable'].astype(int)  #          0/1 

# NOTE: comment translated/cleaned for English-only repository.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# NOTE: comment translated/cleaned for English-only repository.
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
clf.fit(X_train, y_train)

# NOTE: comment translated/cleaned for English-only repository.
y_pred = clf.predict(X_test)

print("(Accuracy):", accuracy_score(y_test, y_pred))
print("\n (Classification Report):\n", classification_report(y_test, y_pred))
print(" (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# NOTE: comment translated/cleaned for English-only repository.
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# NOTE: comment translated/cleaned for English-only repository.
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

weight_ratio = (y == 0).sum() / (y == 1).sum()

# NOTE: comment translated/cleaned for English-only repository.
clf = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=weight_ratio,
    random_state=42
)

# NOTE: comment translated/cleaned for English-only repository.
clf.fit(X_train, y_train)

# NOTE: comment translated/cleaned for English-only repository.
y_pred = clf.predict(X_test)

# NOTE: comment translated/cleaned for English-only repository.
print("    (Accuracy):", accuracy_score(y_test, y_pred))
print("\n     (Classification Report):\n", classification_report(y_test, y_pred))
print("     (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))
# NOTE: comment translated/cleaned for English-only repository.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# NOTE: comment translated/cleaned for English-only repository.
print(" K-Fold      F1-score ")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"   F1-score: {f1_scores.mean():.3f}   {f1_scores.std():.3f}")


In [ ]:
import sys
sys.path.append(r"C:\Users\Adolph\anaconda3\Lib\site-packages")

import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
import shap
import numpy as np

explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)  # shape: [samples, features, classes]

# NOTE: comment translated/cleaned for English-only repository.
shap_values_class1 = shap_values[:, :, 1]  # shape: (1376, 7)

# NOTE: comment translated/cleaned for English-only repository.
shap.summary_plot(
    shap_values_class1,
    features=X_test.values,
    feature_names=X_test.columns,
    plot_type="bar"
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# NOTE: comment translated/cleaned for English-only repository.
models = ['Spatial only', 'Score + Spatial']
accuracy = [0.71, 0.99]
f1_class1 = [0.24, 0.96]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
acc_bar = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='#4CAF50')
f1_bar = ax.bar(x + width/2, f1_class1, width, label='F1-score (valuable)', color='#2196F3')

# NOTE: comment translated/cleaned for English-only repository.
ax.set_ylabel('Score')
ax.set_title('Performance Comparison of Structured Models')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.1)
ax.legend()

for i, val in enumerate(accuracy):
    ax.text(x[i] - width/2, val + 0.02, f"{val:.2f}", ha='center')
for i, val in enumerate(f1_class1):
    ax.text(x[i] + width/2, val + 0.02, f"{val:.2f}", ha='center')

plt.tight_layout()
plt.savefig("structured_model_performance.png", dpi=300)
plt.show()


In [ ]:
def opponent_density_along_path(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    opps = [(row[f'opponentPosX{i}'], row[f'opponentPosY{i}']) for i in range(1,12)]
    
    count = 0
    for ox, oy in opps:
        if ox < 0 or oy < 0:
            continue

        if min(x0, x1) - 2 <= ox <= max(x0, x1) + 2 and min(y0, y1) - 2 <= oy <= max(y0, y1) + 2:
            count += 1
    return count

df['opp_in_path'] = df.apply(opponent_density_along_path, axis=1)


In [ ]:
import numpy as np

def point_to_segment_distance(px, py, x1, y1, x2, y2):
# NOTE: comment translated/cleaned for English-only repository.
    line_mag = (x2 - x1)**2 + (y2 - y1)**2
    if line_mag == 0:
        return np.sqrt((px - x1)**2 + (py - y1)**2)

    u = ((px - x1) * (x2 - x1) + (py - y1) * (y2 - y1)) / line_mag
    u = max(min(u, 1), 0)
    ix = x1 + u * (x2 - x1)
    iy = y1 + u * (y2 - y1)
    return np.sqrt((px - ix)**2 + (py - iy)**2)

def avg_opp_dist_to_pass_path(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    if x0 < 0 or y0 < 0 or x1 < 0 or y1 < 0:
        return np.nan

    opp_dists = []
    for i in range(1, 12):
        ox, oy = row.get(f'opponentPosX{i}', -1), row.get(f'opponentPosY{i}', -1)
        if ox < 0 or oy < 0:
            continue
        dist = point_to_segment_distance(ox, oy, x0, y0, x1, y1)
        opp_dists.append(dist)

    return np.mean(opp_dists) if opp_dists else np.nan
df['avg_opp_dist_to_path'] = df.apply(avg_opp_dist_to_pass_path, axis=1)


In [ ]:
import numpy as np

def calc_pass_opponent_angle(row):
    x0, y0 = row['passedPlayerPosX'], row['passedPlayerPosY']
    x1, y1 = row['receivedPlayerPosX'], row['receivedPlayerPosY']
    
    if x0 < 0 or y0 < 0 or x1 < 0 or y1 < 0:
        return np.nan

    # Compute normalized pass direction vector
    dx_pass, dy_pass = x1 - x0, y1 - y0
    norm_pass = np.sqrt(dx_pass**2 + dy_pass**2)
    if norm_pass == 0:
        return np.nan
    vec_pass = np.array([dx_pass, dy_pass]) / norm_pass

    # Find nearest opponent and compute direction vector
    min_dist = float('inf')
    vec_opp = None
    for i in range(1, 12):
        ox, oy = row.get(f'opponentPosX{i}', -1), row.get(f'opponentPosY{i}', -1)
        if ox < 0 or oy < 0:
            continue
        dx_opp, dy_opp = ox - x0, oy - y0
        dist = np.sqrt(dx_opp**2 + dy_opp**2)
        if 0 < dist < min_dist:
            min_dist = dist
            vec_opp = np.array([dx_opp, dy_opp]) / dist

    if vec_opp is None:
        return np.nan

    # Compute angle (in degrees) between pass and nearest opponent direction
    dot_product = np.dot(vec_pass, vec_opp)
    angle = np.arccos(np.clip(dot_product, -1.0, 1.0))
    return np.degrees(angle)

# Apply to each row
df['angle_pass_vs_nearest_opp'] = df.apply(calc_pass_opponent_angle, axis=1)


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# NOTE: comment translated/cleaned for English-only repository.
features = [

    'opp_in_path','avg_opp_dist_to_path','angle_pass_vs_nearest_opp'
]

# NOTE: comment translated/cleaned for English-only repository.
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

weight_ratio = (y == 0).sum() / (y == 1).sum()

# NOTE: comment translated/cleaned for English-only repository.
clf = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    scale_pos_weight=weight_ratio,
    random_state=42
)

# NOTE: comment translated/cleaned for English-only repository.
clf.fit(X_train, y_train)

# NOTE: comment translated/cleaned for English-only repository.
y_pred = clf.predict(X_test)

# NOTE: comment translated/cleaned for English-only repository.
print("(Accuracy):", accuracy_score(y_test, y_pred))
print("\n(Classification Report):\n", classification_report(y_test, y_pred))
print(" (Confusion Matrix):\n", confusion_matrix(y_test, y_pred))


In [ ]:
import sys
sys.path.append(r"C:\Users\Adolph\anaconda3\Lib\site-packages")

import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd


# NOTE: comment translated/cleaned for English-only repository.
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# NOTE: comment translated/cleaned for English-only repository.
df_model = df[features + ['is_valuable']].dropna()
X = df_model[features]
y = df_model['is_valuable'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# NOTE: comment translated/cleaned for English-only repository.
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

# NOTE: comment translated/cleaned for English-only repository.
y_pred = clf.predict(X_test)
print("       ")
print(classification_report(y_test, y_pred))

# NOTE: comment translated/cleaned for English-only repository.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# NOTE: comment translated/cleaned for English-only repository.
print(" K-Fold      F1-score ")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"   F1-score: {f1_scores.mean():.3f}   {f1_scores.std():.3f}")


In [ ]:
import shap
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# NOTE: comment translated/cleaned for English-only repository.
shap.summary_plot(shap_values[1], X_test)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# NOTE: comment translated/cleaned for English-only repository.
mean_shap = np.abs(shap_values_class1).mean(axis=0)
feature_names = X_test.columns

# NOTE: comment translated/cleaned for English-only repository.
sorted_idx = np.argsort(mean_shap)[::-1]
sorted_names = feature_names[sorted_idx]
sorted_shap = mean_shap[sorted_idx]

# NOTE: comment translated/cleaned for English-only repository.
plt.figure(figsize=(8, 5))
plt.barh(sorted_names, sorted_shap, color='#e63946')  #     
plt.gca().invert_yaxis()
plt.xlabel("Mean |SHAP value|")
plt.title("Feature Importance for Predicting Valuable Passes")
plt.tight_layout()
plt.savefig("shap_custom_bar.png", dpi=300)
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# NOTE: comment translated/cleaned for English-only repository.
df = pd.read_csv("label_annotation_sample (1).csv")

# NOTE: comment translated/cleaned for English-only repository.
features = [
    'pass_speed', 'receivedPlayerId_PassLength', 'pressure_level_num',
    'pressure_direction_num', 'pass_area_num', 'pass_direction_num', 'xT_change',
    'opp_in_path', 'avg_opp_dist_to_path', 'angle_pass_vs_nearest_opp'
]

# NOTE: comment translated/cleaned for English-only repository.
df = df.dropna(subset=features + ['human_label'])

# NOTE: comment translated/cleaned for English-only repository.
X = df[features]
y = df['human_label'].astype(int)

# NOTE: comment translated/cleaned for English-only repository.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# NOTE: comment translated/cleaned for English-only repository.
clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

# NOTE: comment translated/cleaned for English-only repository.
y_pred = clf.predict(X_test)
print("       ")
print(classification_report(y_test, y_pred))

# NOTE: comment translated/cleaned for English-only repository.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1')

# NOTE: comment translated/cleaned for English-only repository.
print(" K-Fold      F1-score ")
for i, score in enumerate(f1_scores):
    print(f"Fold {i+1}: {score:.3f}")
print(f"   F1-score: {f1_scores.mean():.3f}   {f1_scores.std():.3f}")


In [ ]:
import matplotlib.pyplot as plt

# F1 scores from 5-fold CV
folds = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']
f1_scores = [0.923, 0.875, 0.923, 0.923, 0.769]

plt.figure(figsize=(8, 5))
bars = plt.bar(folds, f1_scores, color='#2196F3')
plt.axhline(y=sum(f1_scores)/len(f1_scores), color='gray', linestyle='--', label='Mean F1')
plt.ylim(0, 1.1)
plt.ylabel('F1-score')
plt.title('5-Fold Cross-Validation F1-score')
plt.legend()

# NOTE: comment translated/cleaned for English-only repository.
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{score:.3f}", ha='center')

plt.tight_layout()
plt.savefig("kfold_f1_scores.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# NOTE: comment translated/cleaned for English-only repository.
metrics = ['Precision (0)', 'Recall (0)', 'F1 (0)', 'Precision (1)', 'Recall (1)', 'F1 (1)']
values = [0.82, 1.00, 0.90, 1.00, 0.71, 0.83]
colors = ['#4CAF50'] * 3 + ['#f44336'] * 3
legend_elements = [
    Patch(facecolor='#4CAF50', edgecolor='black', label='Not Valuable'),
    Patch(facecolor='#f44336', edgecolor='black', label='Valuable')
]

# NOTE: comment translated/cleaned for English-only repository.
plt.figure(figsize=(12, 8))
bars = plt.bar(metrics, values, color=colors, edgecolor='black', alpha=0.9)

# NOTE: comment translated/cleaned for English-only repository.
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
             ha='center', va='bottom', fontsize=16, fontweight='bold')

# NOTE: comment translated/cleaned for English-only repository.
plt.legend(handles=legend_elements, loc='upper right', fontsize=16)

# NOTE: comment translated/cleaned for English-only repository.
plt.ylim(0, 1.1)
plt.xticks(rotation=15, fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.title('RandomForest on Human-Labeled Valuable Passes', fontsize=20, fontweight='bold')
plt.ylabel('Score', fontsize=18, fontweight='bold')

# NOTE: comment translated/cleaned for English-only repository.
plt.tight_layout()
plt.savefig("randomforest_humanlabel_metrics_allbold_fixed.png", dpi=300)
plt.show()


In [ ]:
# NOTE: comment translated/cleaned for English-only repository.
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# NOTE: comment translated/cleaned for English-only repository.
if isinstance(shap_values, list):
    shap_to_plot = shap_values[1]
elif shap_values.ndim == 3:
    shap_to_plot = shap_values[:, :, 1]
else:
    shap_to_plot = shap_values

# NOTE: comment translated/cleaned for English-only repository.
shap.summary_plot(shap_to_plot, X_test)
shap.summary_plot(shap_to_plot, X_test, plot_type="bar")

# NOTE: comment translated/cleaned for English-only repository.
shap.dependence_plot("xT_change", shap_to_plot, X_test)
shap.dependence_plot("pass_speed", shap_to_plot, X_test)

# NOTE: comment translated/cleaned for English-only repository.
i = 3
shap.initjs()
shap.plots.force(
    explainer.expected_value[1],
    shap_values[:, :, 1][i],
    X_test.iloc[i]
)

# NOTE: comment translated/cleaned for English-only repository.
shap.summary_plot(shap_to_plot, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_summary_dot.png", dpi=300, bbox_inches='tight')
plt.close()

